In [1]:
# Cell 1 — Imports
import os
import warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')
print('✅ Imports OK')

✅ Imports OK


In [2]:
# Cell 2 — Load & sample data
df = pd.read_pickle('../data/processed/full_dataset.pkl')
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year

df_sample = (df.groupby(['ticker', 'year'])
               .first()
               .reset_index())

print(f'Total transcripts:  {len(df)}')
print(f'Sampled:            {len(df_sample)}')
print(f'Unique companies:   {df["ticker"].nunique()}')

Total transcripts:  13377
Sampled:            4836
Unique companies:   2182


In [3]:
# Cell 3 — Build documents & chunks
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = []
for _, row in df_sample.iterrows():
    text = ' '.join(str(row['transcript']).split()[:1000])
    docs.append(Document(
        page_content=text,
        metadata={
            'ticker':     row['ticker'],
            'date':       str(row['date'].date()),
            'quarter':    row['q'],
            'direction':  int(row['direction']),
            'pct_change': float(row['pct_change'])
        }
    ))

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f'Documents: {len(docs)}')
print(f'Chunks:    {len(chunks)}')

Documents: 4836
Chunks:    69628


In [4]:
# Cell 4 — Build FAISS tiny index (only run once)
# Uses 200 companies, 200 words each = 2.8MB index
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os

if not os.path.exists('../data/processed/faiss_tiny'):
    print('Building tiny index...')
    df_tiny = df.groupby('ticker').first().reset_index().head(200)
    tiny_docs = []
    for _, row in df_tiny.iterrows():
        text = ' '.join(str(row['transcript']).split()[:200])
        tiny_docs.append(Document(
            page_content=text,
            metadata={
                'ticker':    row['ticker'],
                'date':      str(row['date'].date()),
                'quarter':   row['q'],
                'direction': int(row['direction']),
                'pct_change': float(row['pct_change'])
            }
        ))
    tiny_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
    tiny_chunks = tiny_splitter.split_documents(tiny_docs)
    print(f'Chunks: {len(tiny_chunks)}')
    
    embeddings = HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-MiniLM-L6-v2',
        model_kwargs={'device': 'cpu'}
    )
    vs = FAISS.from_documents(tiny_chunks, embeddings)
    vs.save_local('../data/processed/faiss_tiny')
    print('✅ Tiny index built and saved!')
else:
    print('✅ Tiny index already exists, skipping build')

✅ Tiny index already exists, skipping build


In [5]:
# Cell 5 — Load embeddings + index + Groq client
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

print('Loading embeddings...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)

print('Loading index...')
vs = FAISS.load_local(
    '../data/processed/faiss_tiny',
    embeddings,
    allow_dangerous_deserialization=True
)

client = Groq(api_key=os.getenv('GROQ_API_KEY'))
print('✅ All loaded!')

Loading embeddings...


/var/folders/1d/9fkc9vwx4ms_mn76lbp712yr0000gn/T/ipykernel_92541/1298865728.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5353.19it/s]


Loading index...
✅ All loaded!


In [6]:
# Cell 6 — Define ask function
def ask(question, k=3):
    docs = vs.similarity_search(question, k=k)
    context = '\n'.join([
        f"[{d.metadata['ticker']} | {d.metadata['date']} | {d.metadata['quarter']}]\n{d.page_content}"
        for d in docs
    ])
    r = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role':'user','content':
            f'Using these earnings calls:\n{context}\n\nQuestion: {question}\nAnswer citing ticker and date:'}],
        max_tokens=400
    )
    answer = r.choices[0].message.content
    print(f'\n❓ {question}')
    print(f'💬 {answer}')
    print(f'\n📄 Sources:')
    for d in docs:
        print(f"   → {d.metadata['ticker']} | {d.metadata['date']} | {d.metadata['quarter']}")
    print('-' * 60)
    return answer

print('✅ ask() ready!')

✅ ask() ready!


In [7]:
# Cell 7 — Test the chatbot
ask('What did companies say about supply chain issues in 2021?')
ask('Which companies mentioned COVID impact on revenue?')
ask('What guidance did tech companies give for 2022?')


❓ What did companies say about supply chain issues in 2021?
💬 Based on the provided earnings calls, there is no mention of supply chain issues in 2021, as the latest date mentioned is 2020-11-06 (ADMA) and 2020-08-06 (ADTN). However, ADTN (2020-08-06) does mention the ability of component supplies to align with customer demand as a risk factor.

📄 Sources:
   → ADTN | 2020-08-06 | 2020-Q2
   → ADM | 2019-10-31 | 2019-Q3
   → ADMA | 2020-11-06 | 2020-Q3
------------------------------------------------------------

❓ Which companies mentioned COVID impact on revenue?
💬 Although the provided texts do not explicitly mention the impact of COVID-19 on revenue, all three companies (APLE, ANET, and AMPH) discussed the effects of COVID-19 on their business. 

Based on the given information:
- APLE (2021-05-07) mentioned the impact of COVID-19 on the company's business and financial condition.
- ANET (2021-02-18) talked about the impact of COVID-19 on their business and product innovation.
- AM

"None of the provided earnings calls offer guidance for 2022. \n\n* AGEN's call is from 2021-03-15, discussing 2020-Q4, with no mention of 2022 guidance.\n* AGCO's call is from 2019-07-30, discussing 2019-Q2, which is too early to provide guidance for 2022.\n* AFYA's call is from 2021-11-22, discussing 2021-Q3, but the text snippet does not mention specific guidance for 2022.\n\nTherefore, based on the given information, there is no direct guidance for 2022 from these tech companies."